# MapReduce Model

## Learning Objectives

By the end of this notebook you will be able to:

1. **Describe** the MapReduce programming model at a conceptual level
2. **Trace** the data flow through Map, Shuffle & Sort, and Reduce phases
3. **Write** Map and Reduce functions for Word Count
4. **Identify** when MapReduce is the right tool for a problem

## Motivation

Modern datasets — web crawls, server logs, genomic sequences — routinely reach terabytes.
A single machine cannot process them in reasonable time.

**Core insight:** most large-scale computations can be expressed as two pure functions applied
in parallel over independent partitions of data, with a grouping step in between.

## The Three Phases

```
Input Splits → [ Map ] → (key, value) pairs → [ Shuffle & Sort ] → grouped values → [ Reduce ] → Output
```

| Phase | Who runs it | What happens |
|-------|------------|--------------|
| **Map** | Programmer | Transform each record into zero or more (key, value) pairs |
| **Shuffle & Sort** | Framework | Group all values by key; sort within each group |
| **Reduce** | Programmer | Aggregate values for each key into a final result |

The programmer writes only Map and Reduce. Everything else — data movement, fault recovery,
task scheduling — is handled by the framework (Hadoop, Spark, etc.).

## Word Count: A Mental Model

Input text distributed across three machines:

```
Split 0: "the cat sat on the mat"
Split 1: "the cat in the hat"
Split 2: "one fish two fish red fish blue fish"
```

**Map phase** — each machine processes its split independently:
```
Split 0 emits: (the,1) (cat,1) (sat,1) (on,1) (the,1) (mat,1)
Split 1 emits: (the,1) (cat,1) (in,1) (the,1) (hat,1)
Split 2 emits: (one,1) (fish,1) (two,1) (fish,1) (red,1) (fish,1) (blue,1) (fish,1)
```

**Shuffle & Sort** — framework groups by key:
```
cat  → [1, 1]
fish → [1, 1, 1, 1]
the  → [1, 1, 1, 1, 1]
...
```

**Reduce phase** — sum the list for each key:
```
(cat, 2)  (fish, 4)  (the, 5)  ...
```

In [1]:
from collections import defaultdict


def map_fn(split_name, text):
    """Emit (word, 1) for every word in the text."""
    for word in text.lower().split():
        word = word.strip(".,!?;:")
        if word:
            yield word, 1


def reduce_fn(word, counts):
    """Sum counts for a word."""
    return word, sum(counts)


def run_mapreduce(splits, map_fn, reduce_fn):
    # Map
    intermediate = []
    for name, text in splits:
        intermediate.extend(map_fn(name, text))

    # Shuffle & Sort
    grouped = defaultdict(list)
    for k, v in intermediate:
        grouped[k].append(v)

    # Reduce
    return [reduce_fn(k, vs) for k, vs in sorted(grouped.items())]


splits = [
    ("split0", "the cat sat on the mat"),
    ("split1", "the cat in the hat"),
    ("split2", "one fish two fish red fish blue fish"),
]

for word, count in run_mapreduce(splits, map_fn, reduce_fn):
    print(f"{word:10s} {count}")

blue       1
cat        2
fish       4
hat        1
in         1
mat        1
on         1
one        1
red        1
sat        1
the        4
two        1


## Data Flow Diagram

```
┌──────────┐     map      ┌──────────────────────────┐   shuffle   ┌──────────────┐   reduce   ┌────────────┐
│ Split 0  │ ──────────▶  │ (the,1)(cat,1)(sat,1)... │ ──────────▶ │ cat: [1,1]   │ ─────────▶ │ (cat,  2)  │
├──────────┤              └──────────────────────────┘             │ fish:[1,1,1,1]│            │ (fish, 4)  │
│ Split 1  │ ──────────▶  │ (the,1)(cat,1)(in,1)...  │ ──────────▶ │ the: [1,1,1,1,1]│         │ (the,  5)  │
├──────────┤              └──────────────────────────┘             │ ...           │            │ ...        │
│ Split 2  │ ──────────▶  │ (one,1)(fish,1)...       │            └──────────────┘            └────────────┘
└──────────┘              └──────────────────────────┘
```

Key properties:
- Map tasks run **in parallel** — one per split
- Reduce tasks run **in parallel** — one per unique key (or key range)
- The only sequential bottleneck is the **shuffle** (network transfer)

## When to Use MapReduce

| Good fit | Poor fit |
|----------|---------|
| One or a few passes over large data | Iterative algorithms needing many passes |
| Aggregations, counting, joins | Low-latency / interactive queries |
| Embarrassingly parallel transforms | Fine-grained inter-task communication |
| Batch ETL pipelines | Algorithms with irregular data dependencies |

The communication cost (total bytes in intermediate pairs) is the dominant cost metric.
Designing a MapReduce algorithm means minimising this cost.